In [1]:
import glob
import pandas as pd

# Load files

In [2]:
files = glob.glob("data/field_labeling_results.jsonl")
files

['data/field_labeling_results.jsonl']

In [3]:
df = pd.concat([pd.read_json(x, lines=True) for x in files]).reset_index(drop=True)
df

,metadata_field,model,raw_output,usage,cost,status,incomplete_details,parsed_output,error
0,geographic_scope,gpt-5.4-mini,"{""canonical_name"":""geographic_scope"",""confiden...","{'input_tokens': 3066, 'input_tokens_details':...","{'input_tokens': 3066, 'output_tokens': 47, 't...",completed,NaN,"{'canonical_name': 'geographic_scope', 'confid...",NaN
1,unit_of_measure,gpt-5.4-mini,"{""canonical_name"":""unit_of_measure"",""confidenc...","{'input_tokens': 3059, 'input_tokens_details':...","{'input_tokens': 3059, 'output_tokens': 38, 't...",completed,NaN,"{'canonical_name': 'unit_of_measure', 'confide...",NaN
2,indicator_name,gpt-5.4-mini,"{""canonical_name"":""indicator_name"",""confidence...","{'input_tokens': 3107, 'input_tokens_details':...","{'input_tokens': 3107, 'output_tokens': 34, 't...",completed,NaN,"{'canonical_name': 'indicator_name', 'confiden...",NaN
3,time_period,gpt-5.4-mini,"{""canonical_name"":""time_period"",""confidence"":""...","{'input_tokens': 3081, 'input_tokens_details':...","{'input_tokens': 3081, 'output_tokens': 51, 't...",completed,NaN,"{'canonical_name': 'time_period', 'confidence'...",NaN
4,row_dimension,gpt-5.4-mini,"{""canonical_name"":""row_dimension"",""confidence""...","{'input_tokens': 3094, 'input_tokens_details':...","{'input_tokens': 3094, 'output_tokens': 49, 't...",completed,NaN,"{'canonical_name': 'row_dimension', 'confidenc...",NaN
...,...,...,...,...,...,...,...,...,...
828,measure_labels,gpt-5.4-mini,"{""canonical_name"":""series_labels"",""confidence""...","{'input_tokens': 3039, 'input_tokens_details':...","{'input_tokens': 3039, 'output_tokens': 377, '...",completed,NaN,"{'canonical_name': 'series_labels', 'confidenc...",NaN
829,model_outcome_variables,gpt-5.4-mini,"{""canonical_name"":""indicator_name"",""confidence...","{'input_tokens': 3034, 'input_tokens_details':...","{'input_tokens': 3034, 'output_tokens': 115, '...",completed,NaN,"{'canonical_name': 'indicator_name', 'confiden...",NaN
830,model_or_fit_type,gpt-5.4-mini,"{""canonical_name"":""analysis_method"",""confidenc...","{'input_tokens': 3049, 'input_tokens_details':...","{'input_tokens': 3049, 'output_tokens': 81, 't...",completed,NaN,"{'canonical_name': 'analysis_method', 'confide...",NaN
831,measured_variable,gpt-5.4-mini,"{""canonical_name"":""indicator_name"",""confidence...","{'input_tokens': 3026, 'input_tokens_details':...","{'input_tokens': 3026, 'output_tokens': 54, 't...",completed,NaN,"{'canonical_name': 'indicator_name', 'confiden...",NaN


# Errors

In [4]:
df[df["error"].notna()]

,metadata_field,model,raw_output,usage,cost,status,incomplete_details,parsed_output,error


# Cost

In [5]:
df = df.reset_index(drop=True).join(pd.json_normalize(df["cost"]))

total = df["total_cost_usd"].sum()
mean = df["total_cost_usd"].mean()
std = df["total_cost_usd"].std()

print(f"Total cost: ${total:.2f}")
print(f"Ave cost: ${mean:.4f}")
print(f"Std dev: ${std:.4f}")

Total cost: $2.70
Ave cost: $0.0032
Std dev: $0.0010


# Output tokens

In [6]:
total = df["output_tokens"].sum()
mean = df["output_tokens"].mean()
std = df["output_tokens"].std()

print(f"Total output tokens: {total}")
print(f"Ave output tokens: {mean}")
print(f"Std dev: {std:.2f}")

Total output tokens: 176437
Ave output tokens: 211.8091236494598
Std dev: 232.35


# Parsed output

In [7]:
res = (
    df.reset_index(drop=True)
    .join(pd.json_normalize(df["parsed_output"]))[
        ["metadata_field", "canonical_name", "confidence"]
    ]
    .copy()
)
res

,metadata_field,canonical_name,confidence
0,geographic_scope,geographic_scope,high
1,unit_of_measure,unit_of_measure,high
2,indicator_name,indicator_name,high
3,time_period,time_period,high
4,row_dimension,row_dimension,high
...,...,...,...
828,measure_labels,series_labels,medium
829,model_outcome_variables,indicator_name,high
830,model_or_fit_type,analysis_method,high
831,measured_variable,indicator_name,high


In [8]:
# Join fields
res = res.merge(pd.read_csv("outputs/3.0-field_profiles.csv"), on="metadata_field")

res

,metadata_field,canonical_name,confidence,count,n_snapshots,support,corpora_count,figure_count,table_count,source_snapshot_count,source_document_count,source_both_count,top_observed_values,n_unique_observed_values,top_description_values,top_reasoning_values
0,geographic_scope,geographic_scope,high,192,192,0.914286,3,96,96,65,56,71,"['Lebanon', 'Djibouti', 'Uganda', 'Croatia', '...",121,['Country or geographic area to which the tabl...,['Geographic scope is essential for country-le...
1,unit_of_measure,unit_of_measure,high,180,180,0.857143,3,98,82,180,0,0,"['Percent (%)', 'Percent', 'US$, Millions', 'U...",129,['Measurement unit used for the plotted values...,['Units are essential for interpreting and reu...
2,indicator_name,indicator_name,high,152,152,0.723810,3,89,63,152,0,0,"['Adjustment functions', 'Forcibly displaced a...",140,['Name of the measured concept or indicator re...,['Indicator names are central for searching an...
3,time_period,time_period,high,149,149,0.709524,3,86,63,142,4,3,"['2023', 'décembre 2023', 'Septembre 2023', '1...",109,['Time period covered by the data in the snaps...,['Time period enables temporal filtering and c...
4,row_dimension,row_dimension,high,102,102,0.485714,3,16,86,102,0,0,"['Territoires', 'Region', 'Expenditure Categor...",96,['The conceptual dimension represented by the ...,['Row dimension clarifies how the table is org...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
828,measure_labels,series_labels,medium,1,1,0.004762,1,0,1,1,0,0,['Credit Amount; Grant Amount; SML Amount; Gua...,1,['Named measures or value columns reported in ...,['Measure labels support precise discovery of ...
829,model_outcome_variables,indicator_name,high,1,1,0.004762,1,0,1,1,0,0,['Per capita expenditure; Per capita income'],1,['The dependent or outcome variables reported ...,['Outcome variables are central for searching ...
830,model_or_fit_type,analysis_method,high,1,1,0.004762,1,1,0,1,0,0,['Quadratic fit whole annual sample; cross-sec...,1,"['Statistical model, fitted curve, or estimati...",['Model information helps users find figures i...
831,measured_variable,indicator_name,high,1,1,0.004762,1,1,0,1,0,0,['Demand volume'],1,['The quantitative variable represented by the...,['Identifying the measured variable improves a...


In [9]:
res.to_csv("4.1-labeled_field_profiles.csv", index=False)